# 🕵️ Dia 10 — O shell Linux, o sistema de arquivos e o modelo de permissões

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 10* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-10.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Segunda-feira, 16h10. O Juliano, do TI da Serra Azul, mandou os logs do portal por uma pasta `/srv/entrega-vigia/` com modo `777`. Um `LEIA-ME.txt` diz que foi modificado em 14/09, mas nasceu hoje.

Rafa: *"Antes de ler uma linha do log, me diz em que você pode confiar."*

Neste notebook você vai:

1. **Traduzir permissões** entre a forma simbólica (`-rw-r-----`) e a octal (`640`), com SUID, SGID e *sticky*.
2. **Refazer o experimento do `touch`** em Python e ver o `ctime` andar sozinho.
3. **Rodar um detector de carimbo mexido** sobre a saída de `stat` da entrega da Serra Azul.
4. **Jogar o quiz do `rwx`** e gerar a ficha de triagem da entrega.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. Permissões: de `rwx` para octal e de volta

Cada trio `rwx` vira um dígito somando `r`=4, `w`=2, `x`=1. Os bits especiais ficam num quarto dígito, à esquerda: SUID=4000, SGID=2000, *sticky*=1000. Eles aparecem **no lugar do `x`**: `s` (com execução) ou `S` (sem execução) no dono/grupo, `t`/`T` nos outros.

In [ ]:
# ------------------------------------------------------------
# SIMBÓLICO -> OCTAL
# Ex.: "-rwsr-xr-x" (o /usr/bin/passwd) -> "4755"
# ------------------------------------------------------------

def simbolico_para_octal(modo: str) -> str:
    """Converte '-rw-r-----' (10 caracteres, como no ls -l) em '640' ou '4755'."""
    if len(modo) != 10:                       # 1 de tipo + 9 de permissão
        raise ValueError("Use os 10 caracteres do ls -l, ex.: -rw-r-----")
    trios = [modo[1:4], modo[4:7], modo[7:10]]   # dono, grupo, outros
    especiais = 0                             # o "quarto dígito" (SUID/SGID/sticky)
    digitos = []
    for posicao, trio in enumerate(trios):
        valor = 0
        if trio[0] == "r":                    # leitura vale 4
            valor += 4
        if trio[1] == "w":                    # escrita vale 2
            valor += 2
        x = trio[2]                           # a 3ª letra pode ser x, -, s, S, t, T
        if x in "xst":                        # minúscula = execução ligada
            valor += 1
        if x in "sS":                         # s/S no dono = SUID; no grupo = SGID
            especiais += 4 if posicao == 0 else 2
        if x in "tT":                         # t/T nos outros = sticky bit
            especiais += 1
        digitos.append(str(valor))
    octal = "".join(digitos)
    return (str(especiais) + octal) if especiais else octal   # só mostra o 4º dígito se existir


# Os exemplos do dia
for m in ["-rw-r-----", "-rwsr-xr-x", "drwxrwxrwt", "drwxrwxrwx", "-rw-rw-r--"]:
    print(f"{m}  ->  {simbolico_para_octal(m)}")

In [ ]:
# ------------------------------------------------------------
# OCTAL -> SIMBÓLICO (o caminho inverso, útil para conferir chmod)
# ------------------------------------------------------------

def octal_para_simbolico(octal: str, tipo: str = "-") -> str:
    """'640' -> '-rw-r-----'; '1777' com tipo 'd' -> 'drwxrwxrwt'."""
    octal = octal.zfill(4)                    # garante 4 dígitos: '640' vira '0640'
    especiais = int(octal[0])                 # 4=SUID, 2=SGID, 1=sticky (podem somar)
    saida = tipo
    for posicao, d in enumerate(octal[1:]):
        d = int(d)
        r = "r" if d & 4 else "-"             # & = "este bit está ligado?"
        w = "w" if d & 2 else "-"
        x_ligado = bool(d & 1)
        x = "x" if x_ligado else "-"
        # troca o x pelo bit especial da posição, se houver
        if posicao == 0 and especiais & 4:
            x = "s" if x_ligado else "S"
        if posicao == 1 and especiais & 2:
            x = "s" if x_ligado else "S"
        if posicao == 2 and especiais & 1:
            x = "t" if x_ligado else "T"
        saida += r + w + x
    return saida


print(octal_para_simbolico("640"))
print(octal_para_simbolico("4755"))
print(octal_para_simbolico("1777", tipo="d"))   # o /tmp

## 2. O experimento do `touch`, em Python

O `os.utime` faz o mesmo que `touch -d`: muda `atime` e `mtime` para o valor que você quiser. O `ctime` você **não** consegue escolher: o próprio kernel o empurra para "agora" quando você mexe nos outros carimbos.

In [ ]:
import os, time
from datetime import datetime, timezone
from pathlib import Path

def carimbos(p):
    """Devolve mtime e ctime em texto UTC, com milissegundos."""
    s = os.stat(p)                                       # lê o inode
    f = lambda ns: datetime.fromtimestamp(ns / 1e9, timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    return f"mtime={f(s.st_mtime_ns)}  ctime={f(s.st_ctime_ns)}"

Path("lab-dia10").mkdir(exist_ok=True)
ev = Path("lab-dia10/evidence.txt")
ev.write_text("original content\n")                     # cria o arquivo: os três carimbos nascem juntos
print("1) criado:          ", carimbos(ev))

time.sleep(1.2)                                          # espera um pouco para o relógio andar

alvo = datetime(2019, 6, 1, 12, 0, 0, tzinfo=timezone.utc).timestamp()
os.utime(ev, (alvo, alvo))                               # = touch -d '2019-06-01 12:00:00'
print("2) depois do touch: ", carimbos(ev))

time.sleep(1.2)
os.chmod(ev, 0o600)                                      # muda só a permissão
print("3) depois do chmod: ", carimbos(ev))
print("\nRepare: o mtime 'voltou' a 2019, mas o ctime só andou para a frente.")

## 3. A entrega da Serra Azul: detector de carimbo mexido

A célula abaixo carrega a saída de `stat` que o Juliano mandou ([`caso-continuo/dados/dia-10/stat-entrega-ti.txt`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/caso-continuo/dados/dia-10/stat-entrega-ti.txt)) e aplica três regras:

1. `mtime` **anterior** ao `birth` → o conteúdo "mudou" antes de o arquivo existir;
2. `ctime` muito depois do `mtime` → alguém mexeu em metadados depois do último conteúdo;
3. quem pode escrever: diretório ou arquivo gravável pelos "outros" ou por um grupo de serviço.

In [ ]:
# ------------------------------------------------------------
# Carrega o arquivo do caso: primeiro tenta o repositório no GitHub,
# depois cai na cópia embutida (offline-first: o notebook nunca trava).
# ------------------------------------------------------------
import urllib.request                      # cliente HTTP da biblioteca padrão

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-10/stat-entrega-ti.txt"
EMBUTIDO = "# DADOS FICTÍCIOS — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, empresa ou IP real.\n# Saída de `stat` que o TI da Serra Azul (Juliano Prates) colou no e-mail de entrega,\n# rodada por ele no servidor sa-web01, a pedido da Vigia, logo antes do envio (16h05 BRT de 5/out).\n# O servidor grava logs em -03:00, mas a Vigia pediu `TZ=UTC stat`, por isso aqui tudo sai em +0000.\n# Formato: stat -c '%n|%U:%G|%a|%A|mtime=%y|ctime=%z|birth=%w'\n/var/log/nginx/access.log|www-data:adm|640|-rw-r-----|mtime=2026-10-05 18:58:41.120 +0000|ctime=2026-10-05 18:58:41.120 +0000|birth=2026-10-05 03:00:02.410 +0000\n/var/log/nginx/access.log.1|www-data:adm|640|-rw-r-----|mtime=2026-10-05 02:59:58.907 +0000|ctime=2026-10-05 03:00:02.399 +0000|birth=2026-10-04 03:00:01.882 +0000\n/var/log/auth.log|syslog:adm|640|-rw-r-----|mtime=2026-10-05 18:55:10.004 +0000|ctime=2026-10-05 18:55:10.004 +0000|birth=2026-10-04 03:00:01.311 +0000\n/srv/entrega-vigia/access.log.1|juliano:juliano|664|-rw-rw-r--|mtime=2026-10-05 19:02:17.551 +0000|ctime=2026-10-05 19:02:17.551 +0000|birth=2026-10-05 19:02:17.549 +0000\n/srv/entrega-vigia/auth.log|juliano:juliano|664|-rw-rw-r--|mtime=2026-10-05 19:02:17.560 +0000|ctime=2026-10-05 19:02:17.560 +0000|birth=2026-10-05 19:02:17.558 +0000\n/srv/entrega-vigia/LEIA-ME.txt|juliano:juliano|664|-rw-rw-r--|mtime=2026-09-14 12:00:00.000 +0000|ctime=2026-10-05 19:05:40.212 +0000|birth=2026-10-05 19:04:11.020 +0000\n/srv/entrega-vigia|juliano:www-data|777|drwxrwxrwx|mtime=2026-10-05 19:05:40.212 +0000|ctime=2026-10-05 19:05:40.212 +0000|birth=2026-10-05 19:01:55.300 +0000\n"

try:
    with urllib.request.urlopen(URL, timeout=8) as resp:   # timeout curto: sem rede, desiste logo
        STAT_TXT = resp.read().decode("utf-8")
    origem = "GitHub"
except Exception as erro:                  # sem internet, repo privado, URL mudou...
    STAT_TXT = EMBUTIDO                   # ...plano B: a cópia que veio dentro do notebook
    origem = f"cópia embutida ({type(erro).__name__})"

print(f"Carregado de: {origem} — {len(STAT_TXT.splitlines())} linhas")

In [ ]:
from datetime import datetime, timedelta

def ler_data(txt):
    """'2026-10-05 19:02:17.551 +0000' -> datetime (ignora o fuso: todas estão em +0000)."""
    return datetime.strptime(txt[:23], "%Y-%m-%d %H:%M:%S.%f")

linhas = []
for bruta in STAT_TXT.splitlines():
    if not bruta.strip() or bruta.startswith("#"):       # pula comentários e linhas vazias
        continue
    caminho, dono_grupo, octal, simb, m, c, b = bruta.split("|")
    linhas.append({
        "caminho": caminho, "dono_grupo": dono_grupo, "octal": octal, "simb": simb,
        "mtime": ler_data(m.split("=")[1]),             # "mtime=..." -> só a data
        "ctime": ler_data(c.split("=")[1]),
        "birth": ler_data(b.split("=")[1]),
    })

alertas = {}
for l in linhas:
    a = []
    if l["mtime"] < l["birth"] - timedelta(seconds=1):  # 1 s de folga para arredondamentos
        a.append("mtime ANTERIOR ao birth (touch -d? ou cópia com cp -p?)")
    if l["ctime"] - l["mtime"] > timedelta(hours=1):
        a.append("ctime muito depois do mtime (metadado mexido depois do conteúdo)")
    if l["simb"][8] == "w":                              # posição 8 = 'w' dos outros
        a.append("GRAVÁVEL POR TODOS (o+w)")
    dono, grupo = l["dono_grupo"].split(":")
    if l["simb"][5] == "w" and grupo in ("www-data", "nginx", "apache"):
        a.append(f"grupo de serviço '{grupo}' pode escrever")
    alertas[l["caminho"]] = a
    print(f"{l['simb']} {l['octal']:>4} {l['dono_grupo']:<18} {l['caminho']}")
    for x in a:
        print("    ⚠️ ", x)

> 💡 **Leia com calma o `access.log.1` original:** `mtime` às 02:59:58 e o `access.log` atual nasce às 03:00:02 UTC. Isso não é alerta, é o `logrotate` rodando à meia-noite do horário de Brasília (00h BRT = 03h UTC). Saber o que é **normal** é metade da triagem.
>
> 🧠 **Camada 8:** o detector marcou o `LEIA-ME.txt`. Isso **não** prova que o Juliano adulterou nada: `cp -p` preserva o `mtime` do original e produz exatamente esse padrão. Escreva a explicação banal antes da maliciosa.

## 4. 🎮 Quiz do `rwx`

Oito perguntas. Responda com a **letra** da opção. Quer só ler o gabarito? Mude `MODO_JOGO` para `False`.

In [ ]:
MODO_JOGO = True   # False = só mostra as respostas, sem perguntar

QUIZ = [
    ("Quanto vale -rw-r----- em octal?", ["a) 640", "b) 644", "c) 460"], "a",
     "rw- = 4+2 = 6; r-- = 4; --- = 0."),
    ("O que o 'w' num DIRETÓRIO permite?", ["a) Editar qualquer arquivo dentro dele", "b) Criar e apagar arquivos dentro dele, de qualquer dono", "c) Nada, só vale para arquivos"], "b",
     "Escrita no diretório = mexer na lista de nomes. Por isso o sticky bit existe no /tmp."),
    ("Qual carimbo o 'touch -d' NÃO consegue escolher?", ["a) mtime", "b) atime", "c) ctime"], "c",
     "Mudar mtime/atime atualiza o inode, e isso empurra o ctime para agora."),
    ("O ctime é a hora de criação do arquivo?", ["a) Sim", "b) Não, é a última mudança do inode", "c) Só no ext4"], "b",
     "Criação, quando existe, é o birth (%w no stat)."),
    ("Por que o atime é evidência fraca?", ["a) Porque com relatime ele quase não é atualizado", "b) Porque é sempre UTC", "c) Porque só o root vê"], "a",
     "relatime só atualiza o atime de vez em quando; ausência de leitura não se prova com ele."),
    ("-rwsr-xr-x é...", ["a) SGID", "b) SUID", "c) sticky"], "b",
     "O 's' no trio do DONO é SUID (4000)."),
    ("Na entrega da Serra Azul, /srv/entrega-vigia tem modo 777. Quem podia trocar um log antes de você baixar?", ["a) Só o Juliano", "b) Qualquer processo do servidor, inclusive o nginx", "c) Ninguém, os arquivos são 664"], "b",
     "Com w para todos no diretório, qualquer um apaga e recria o arquivo, apesar do 664 dele."),
    ("LEIA-ME.txt: mtime 14/09, birth 05/10. O que você escreve no relatório?", ["a) 'O funcionário adulterou a data'", "b) 'Inconsistência de carimbos; compatível com touch -d ou cópia com cp -p; perguntar ao TI'", "c) Nada, é irrelevante"], "b",
     "Inconsistência é uma pergunta, não um veredito. Explicação banal antes da maliciosa."),
]

acertos = 0
for i, (pergunta, opcoes, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {pergunta}")
    for o in opcoes:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b", "c"):
            r = input("   Sua resposta (a/b/c): ").strip().lower()
        ok = (r == certa)
        acertos += ok
        print("   ✅ Isso!" if ok else f"   ❌ Era {certa}.")
    print("   Por quê:", porque)

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 6 else "— Meta do Checkpoint BR: 6.")

## 5. 📝 A ficha de triagem da entrega

Preencha os campos do formulário e rode. A célula gera `notas/isca-dia10-ficha-entrega.md` com as sete linhas, os alertas da seção 3 e a sua conclusão.

In [ ]:
#@title Ficha de triagem da entrega { display-mode: "form" }
pergunta_ao_ti = "Juliano, como os arquivos foram copiados para /srv/entrega-vigia (cp, cp -p, rsync, scp)? E o LEIA-ME.txt veio de onde?"  #@param {type:"string"}
recomendacao = "Próxima entrega: diretório 750 (juliano, sem www-data) e sha256sum calculado no servidor ANTES da cópia, enviado no corpo do e-mail."  #@param {type:"string"}
conclusao = "Os carimbos dos logs são compatíveis com a rotação diária às 03:00 UTC. A pasta de entrega era gravável por qualquer processo do servidor; sem hash na origem, a integridade da cópia depende da palavra do TI. O LEIA-ME.txt tem mtime anterior ao birth, causa não determinada."  #@param {type:"string"}

from pathlib import Path
from datetime import datetime, timezone

agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
cab = "| Caminho | Dono:Grupo | Modo (octal / simbólico) | mtime (UTC) | ctime (UTC) | birth (UTC) | Alertas | SHA-256 |\n|---|---|---|---|---|---|---|---|\n"
corpo = ""
for l in linhas:                                   # 'linhas' e 'alertas' vêm da seção 3
    al = "; ".join(alertas[l["caminho"]]) or "consistente"
    corpo += (f"| `{l['caminho']}` | {l['dono_grupo']} | {l['octal']} / `{l['simb']}` | "
              f"{l['mtime']:%Y-%m-%d %H:%M:%S} | {l['ctime']:%Y-%m-%d %H:%M:%S} | {l['birth']:%Y-%m-%d %H:%M:%S} | {al} | pendente |\n")

ficha = f"""# Ficha de triagem — entrega de logs da Serra Azul (Dia 10)

Gerada em (UTC): {agora}
Fonte: stat-entrega-ti.txt (saída de stat enviada pelo TI da cooperativa)

{cab}{corpo}
## Pergunta ao TI
{pergunta_ao_ti}

## Recomendação
{recomendacao}

## Conclusão
{conclusao}
"""
Path("notas").mkdir(exist_ok=True)
saida = Path("notas/isca-dia10-ficha-entrega.md")
saida.write_text(ficha, encoding="utf-8")
print(ficha)
print("💾 Salvo em", saida)

try:
    from google.colab import files                  # só existe no Colab
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou na pasta notas/)")

### (Opcional) No seu Linux de verdade

Se estiver numa máquina Linux (ou no próprio Colab, que roda Ubuntu), estas linhas refazem o Passo 4 do README. O teste automático pula linhas com `!`.

In [ ]:
# Opcional: só roda no Colab/Linux. Linhas com "!" são comandos de shell.
!stat -c '%n|%U:%G|%a|mtime=%y|ctime=%z|birth=%w' /etc/passwd /etc/shadow /usr/bin/passwd
!find / -xdev -type f -perm -4000 2>/dev/null | sort | head -20
print("(Se você está vendo só esta linha, os comandos de shell foram pulados.)")

## ✅ Antes de fechar o Dia 10

- [ ] Quiz com pelo menos 6/8
- [ ] `notas/isca-dia10-ficha-entrega.md` gerado e com a pergunta ao TI
- [ ] `day10-triage-card.md` e `suid-baseline.txt` feitos na sua máquina Linux (README, Passos 1 a 5)
- [ ] `status` do dia 10 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 11):** triagem de log web com `grep`, `awk` e `sed`. O `access.log` do portal real da Serra Azul vai contar quem copiou o site e quando.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>